In [9]:
import torch
import torch.nn as nn
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import FeaturePyramidNetwork
from torchvision.models.detection.rpn import RPNHead
from torchvision.models.detection.rpn import RPNHead, RegionProposalNetwork


class SEBlock(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()

        hidden_channels = max(channels // reduction, 1)

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, hidden_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden_channels, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)


class DynamicDilationSERPNHead(RPNHead):
    def __init__(self, in_channels, num_anchors, reduction=4):
        super().__init__(in_channels, num_anchors)

     
        self.dilated1 = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=3, padding=1, dilation=1
        )

        self.dilated2 = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=3, padding=2, dilation=2
        )

        self.dilated3 = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=3, padding=3, dilation=3
        )

      
        self.se = SEBlock(in_channels, reduction=reduction)

        
        self.cls_logits = nn.Conv2d(
            in_channels, num_anchors, kernel_size=1
        )

        self.bbox_pred = nn.Conv2d(
            in_channels, num_anchors * 4, kernel_size=1
        )

        self._initialize_weights()

    def _initialize_weights(self):
        layers = [
            self.dilated1,
            self.dilated2,
            self.dilated3,
            self.cls_logits,
            self.bbox_pred
        ]

        for layer in layers:
            nn.init.normal_(layer.weight, std=0.01)
            nn.init.constant_(layer.bias, 0)

    def forward(self, x):
        logits = []
        bbox_reg = []

        for feature in x:
          
            f1 = F.relu(self.dilated1(feature))
            f2 = F.relu(self.dilated2(feature))
            f3 = F.relu(self.dilated3(feature))

            
            fused = f1 + f2 + f3

          
            fused = self.se(fused)

           
            logits.append(self.cls_logits(fused))
            bbox_reg.append(self.bbox_pred(fused))

        return logits, bbox_reg

In [10]:
# Developing customised Faster_RCNN model with parallel dilated convolution and SE attention
def get_model(num_classes):
    
    model = fasterrcnn_resnet50_fpn(pretrained=True)
    
  
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    anchor_generator = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,), (512,)),  
        aspect_ratios=((0.5, 1.0, 2.0),) * 5          
    )

  
    rpn_head = DynamicDilationSERPNHead(in_channels=256, num_anchors=anchor_generator.num_anchors_per_location()[0])

   
    model.rpn = RegionProposalNetwork(
        anchor_generator,
        rpn_head,
        fg_iou_thresh=0.7,
        bg_iou_thresh=0.3,
        batch_size_per_image=256,
        positive_fraction=0.5,
        pre_nms_top_n=dict(training=2000, testing=1000),
        post_nms_top_n=dict(training=2000, testing=1000),
        nms_thresh=0.7
    )

    return model

num_classes = 3 #provide the number of classes for your dataset
model = get_model(num_classes)
print(model)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(